# Task 2.1 — Silver: Discussion Posts Parsed
## Notebook: 04_silver_discussion_parsed

**Pipeline:** EduPath Learning Analytics · Medallion Architecture
**Layer:** Silver (parsed + computed from Bronze)

**What this notebook does:**
- Reads `discussion_bronze` (raw_json field) from Bronze layer
- Parses raw_json using from_json() with explicit StructType schema
- Computes `post_depth`: 0 = top-level, 1 = reply, 2 = reply-to-reply
- Computes `thread_post_count`: total posts per thread
- Computes `instructor_reply_rate`: instructor posts / student posts per thread
- Writes to `silver.discussion_posts_parsed` — feeds Task 2.3 engagement scoring

**Source table:** `mini_project_grp6.bronze.discussion_bronze`
**Target table:** `mini_project_grp6.silver.discussion_posts_parsed`

**Business Rules enforced:**
- post_depth must be 0, 1, or 2 only
- thread_post_count must be >= 1
- instructor_reply_rate must be between 0.0 and 1.0
- course_id and author_student_id must not be NULL

**Run order:** Run all cells top-to-bottom. Safe to re-run (idempotent — OVERWRITE mode).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
SILVER  = "silver"

# --- Source and target table names (fully qualified) ---
DISCUSSION_BRONZE_TABLE = f"{CATALOG}.{BRONZE}.discussion_bronze"
DISCUSSION_SILVER_TABLE = f"{CATALOG}.{SILVER}.discussion_posts_parsed"
DQ_AUDIT_TABLE          = f"{CATALOG}.audit.dq_audit"

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema for this session ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(SILVER)

print(f"✅ Catalog : {CATALOG}")
print(f"✅ Schema  : {SILVER}")
print(f"✅ Source  : {DISCUSSION_BRONZE_TABLE}")
print(f"✅ Target  : {DISCUSSION_SILVER_TABLE}")

**Expected output:**
- ✅ Catalog : mini_project_grp6
- ✅ Schema  : silver
- ✅ Source  : mini_project_grp6.bronze.discussion_bronze
- ✅ Target  : mini_project_grp6.silver.discussion_posts_parsed

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    BooleanType, TimestampType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Read Bronze + Validate Source

Before parsing, verify that discussion_bronze exists and has data.
We also do a quick source quality check before spending compute on parsing.

In [0]:
# ============================================================
# CELL 5 — Read discussion_bronze and validate before parsing
#
# We read both the raw_json field (for from_json parsing)
# and the pre-extracted top-level columns (for post_depth logic
# which needs parent_post_id relationships)
# ============================================================

# Read the full bronze table
discussion_bronze_df = spark.table(DISCUSSION_BRONZE_TABLE)

total_rows    = discussion_bronze_df.count()
null_raw_json = discussion_bronze_df.filter(F.col("raw_json").isNull()).count()

print("── Source validation: discussion_bronze ──────────")
print(f"   Total rows       : {total_rows:,}")
print(f"   Columns          : {len(discussion_bronze_df.columns)}")
print(f"   NULL raw_json    : {null_raw_json:,}")
print()

# Show column list to confirm raw_json is present
print("── Columns available ─────────────────────────────")
print(discussion_bronze_df.columns)
print()

# Quick sanity check — raw_json sample
print("── raw_json preview (first 3 rows) ───────────────")
discussion_bronze_df.select(
    "post_id",
    F.substring("raw_json", 1, 100).alias("raw_json_preview")
).show(3, truncate=False)

###Expected output:
```
── Source validation: discussion_bronze ──────────
-    Total rows       : 2,000
-    Columns          : 16
-    NULL raw_json    : 0

── Columns available ───────────────────────────── 
- ['post_id', 'thread_id', 'course_id', 'author_student_id',
 'parent_post_id', 'post_ts', 'content_length_chars',
 'has_attachment', 'like_count', 'reply_count',
 'is_instructor_post', 'raw_json', '_source_file',
 '_load_ts', '_last_modified_ts', '_schema_version']
```

## Part B — Parse raw_json with Explicit StructType Schema

We use from_json() with a fully defined StructType.
This is the correct Silver pattern — never use schema_of_json() on production data
because it infers from a single sample row and can miss fields or miscast types.

All parsed fields come from the raw_json column.
The result is a clean, typed Silver table with no string-JSON columns.

In [0]:
# ============================================================
# CELL 7 — Define the StructType schema used by from_json()
#
# This schema mirrors exactly what is stored in raw_json.
# All field names and types must match the Bronze JSON structure.
# If a field is missing in a row, from_json() returns NULL for it
# (safe — we handle NULLs in downstream logic).
# ============================================================

discussion_json_schema = StructType([
    StructField("post_id",              StringType(),    nullable=True),
    StructField("thread_id",            StringType(),    nullable=True),
    StructField("course_id",            StringType(),    nullable=True),
    StructField("author_student_id",    StringType(),    nullable=True),
    StructField("parent_post_id",       StringType(),    nullable=True),
    StructField("post_ts",              TimestampType(), nullable=True),
    StructField("content_length_chars", IntegerType(),   nullable=True),
    StructField("has_attachment",       BooleanType(),   nullable=True),
    StructField("like_count",           IntegerType(),   nullable=True),
    StructField("reply_count",          IntegerType(),   nullable=True),
    StructField("is_instructor_post",   BooleanType(),   nullable=True),
])

print("✅ from_json StructType schema defined")
print(f"   Fields: {len(discussion_json_schema.fields)}")
for f in discussion_json_schema.fields:
    print(f"   {f.name:<30} {str(f.dataType)}")

**Expected output:**

✅ from_json StructType schema defined
Fields: 11

```
post_id                        StringType()
thread_id                      StringType()
course_id                      StringType()
author_student_id              StringType()
parent_post_id                 StringType()
post_ts                        TimestampType()
content_length_chars           IntegerType()
has_attachment                 BooleanType()
like_count                     IntegerType()
reply_count                    IntegerType()
is_instructor_post             BooleanType()
```

In [0]:
# ============================================================
# CELL 8 — Parse raw_json using from_json() + flatten struct fields
#
# Step 1: from_json() parses the raw_json string into a nested struct column
# Step 2: We select each sub-field out of the struct using dot notation
# Step 3: We keep _load_ts and _source_file from bronze for lineage tracking
#
# Why not use the already-parsed top-level Bronze columns directly?
# The Bronze columns were typed on ingestion but raw_json is the source of truth.
# Parsing from raw_json in Silver ensures we can evolve the Silver schema
# independently of Bronze column definitions.
# ============================================================

# Step 1 — Apply from_json() to create a parsed struct column
parsed_struct_df = (
    discussion_bronze_df
    .withColumn(
        "parsed",
        F.from_json(F.col("raw_json"), discussion_json_schema)
    )
)

# Step 2 — Flatten: pull each field out of the struct into its own column
parsed_flat_df = (
    parsed_struct_df
    .select(
        # Parsed fields from raw_json
        F.col("parsed.post_id").alias("post_id"),
        F.col("parsed.thread_id").alias("thread_id"),
        F.col("parsed.course_id").alias("course_id"),
        F.col("parsed.author_student_id").alias("author_student_id"),
        F.col("parsed.parent_post_id").alias("parent_post_id"),
        F.col("parsed.post_ts").alias("post_ts"),
        F.col("parsed.content_length_chars").alias("content_length_chars"),
        F.col("parsed.has_attachment").alias("has_attachment"),
        F.col("parsed.like_count").alias("like_count"),
        F.col("parsed.reply_count").alias("reply_count"),
        F.col("parsed.is_instructor_post").alias("is_instructor_post"),
        # Bronze lineage columns carried forward
        F.col("_source_file"),
        F.col("_load_ts").alias("bronze_load_ts"),
    )
)

parse_count = parsed_flat_df.count()
null_after_parse = parsed_flat_df.filter(F.col("post_id").isNull()).count()

print("── from_json() parsing result ────────────────────")
print(f"   Rows parsed     : {parse_count:,}")
print(f"   NULL post_id    : {null_after_parse}  (malformed JSON rows — expected 0)")
print()
parsed_flat_df.printSchema()

## Part C — Compute post_depth

post_depth tells us how deep in a thread a post is:
- depth 0 = top-level post (parent_post_id IS NULL)
- depth 1 = direct reply to a top-level post
- depth 2 = reply to a reply (deepest level we track)

Why cap at 2?
The architecture spec defines 3 levels for instructor_reply_rate calculation.
Threads deeper than level 2 are rare and treated as level 2 for scoring purposes.

Logic:
- parent_post_id IS NULL                              → depth 0
- parent_post_id exists AND that parent has depth 0  → depth 1
- parent_post_id exists AND that parent has depth 1+  → depth 2

In [0]:
# ============================================================
# CELL 10 — Compute post_depth via self-join
#
# Approach:
#   1. All posts with NULL parent_post_id → depth 0
#   2. Join each post to its parent to find the parent's depth
#   3. depth = min(parent_depth + 1, 2) to cap at level 2
#
# We use a two-level self-join which handles the data we have.
# For deeper threads the depth is capped at 2 per spec.
# ============================================================

# Step 1 — Identify top-level posts (depth 0)
top_level_df = (
    parsed_flat_df
    .filter(F.col("parent_post_id").isNull())
    .select("post_id")
    .withColumn("resolved_depth", F.lit(0))
)

# Step 2 — First-level replies: posts whose parent IS a top-level post
depth_1_df = (
    parsed_flat_df
    .filter(F.col("parent_post_id").isNotNull())
    .join(
        top_level_df.select(F.col("post_id").alias("parent_id")),
        parsed_flat_df["parent_post_id"] == F.col("parent_id"),
        how="inner"
    )
    .select(parsed_flat_df["post_id"])
    .withColumn("resolved_depth", F.lit(1))
)

# Step 3 — Second-level replies: posts whose parent is a depth-1 post
depth_2_df = (
    parsed_flat_df
    .filter(F.col("parent_post_id").isNotNull())
    .join(
        depth_1_df.select(F.col("post_id").alias("parent_id")),
        parsed_flat_df["parent_post_id"] == F.col("parent_id"),
        how="inner"
    )
    .select(parsed_flat_df["post_id"])
    .withColumn("resolved_depth", F.lit(2))
)

# Step 4 — Combine all depth assignments
all_depths_df = top_level_df.union(depth_1_df).union(depth_2_df)

# Step 5 — Join depth back to the main parsed DataFrame
# Use left join — any post not matched (edge case) gets NULL depth → flagged in DQ
parsed_with_depth_df = (
    parsed_flat_df
    .join(all_depths_df, on="post_id", how="left")
)

print("── post_depth distribution ───────────────────────")
parsed_with_depth_df.groupBy("resolved_depth").count().orderBy("resolved_depth").show()

null_depth_count = parsed_with_depth_df.filter(F.col("resolved_depth").isNull()).count()
print(f"   NULL depth (unresolved posts): {null_depth_count}")
print()

# For any unresolved posts (should be 0), default to depth 2 as safe fallback
parsed_with_depth_df = parsed_with_depth_df.withColumn(
    "post_depth",
    F.coalesce(F.col("resolved_depth"), F.lit(2))
).drop("resolved_depth")

print("── Final post_depth distribution ────────────────")
parsed_with_depth_df.groupBy("post_depth").count().orderBy("post_depth").show()

**Expected output:**

```
── post_depth distribution ───────────────────────
+--------------+-----+
|resolved_depth|count|
+--------------+-----+
|0             |~600 |
|1             |~900 |
|2             |~500 |
+--------------+-----+

   NULL depth (unresolved posts): 0

── Final post_depth distribution ────────────────
+----------+-----+
|post_depth|count|
+----------+-----+
|0         |~600 |
|1         |~900 |
|2         |~500 |
+----------+-----+
```

---


## Part D — Compute thread_post_count and instructor_reply_rate

thread_post_count: total number of posts in each thread_id
- Used in engagement scoring to measure discussion activity per thread

instructor_reply_rate: fraction of posts in each thread written by instructors
- Formula: count(is_instructor_post = true) / count(*) per thread_id
- Range: 0.0 to 1.0
- A rate of 0.0 means no instructor participation in that thread
- Used in gold.course_performance_daily as instructor_response_rate

In [0]:
# ============================================================
# CELL 12 — Window aggregations for thread-level metrics
#
# We use Window functions partitioned by thread_id so that
# each row keeps its own columns AND gets the thread-level
# aggregates added as new columns.
#
# This avoids a GROUP BY + join pattern which would be
# more verbose and create an extra shuffle stage.
# ============================================================

# Define window partitioned by thread_id
thread_window = Window.partitionBy("thread_id")

parsed_with_thread_metrics_df = (
    parsed_with_depth_df
    # Total post count per thread
    .withColumn(
        "thread_post_count",
        F.count("post_id").over(thread_window)
    )
    # Count of instructor posts per thread
    .withColumn(
        "instructor_post_count",
        F.sum(
            F.when(F.col("is_instructor_post") == True, 1).otherwise(0)
        ).over(thread_window)
    )
    # instructor_reply_rate = instructor posts / total posts in thread
    # Round to 4 decimal places; avoid divide-by-zero with nullif
    .withColumn(
        "instructor_reply_rate",
        F.round(
            F.col("instructor_post_count") / F.col("thread_post_count"),
            4
        )
    )
    # Drop the intermediate column
    .drop("instructor_post_count")
)

print("── thread_post_count stats ───────────────────────")
parsed_with_thread_metrics_df.select(
    F.min("thread_post_count").alias("min_posts_per_thread"),
    F.max("thread_post_count").alias("max_posts_per_thread"),
    F.round(F.avg("thread_post_count"), 2).alias("avg_posts_per_thread")
).show()

print("── instructor_reply_rate stats ───────────────────")
parsed_with_thread_metrics_df.select(
    F.min("instructor_reply_rate").alias("min_rate"),
    F.max("instructor_reply_rate").alias("max_rate"),
    F.round(F.avg("instructor_reply_rate"), 4).alias("avg_rate")
).show()

print("── sample: one thread with all computed columns ──")
parsed_with_thread_metrics_df.filter(
    F.col("thread_id") == parsed_with_thread_metrics_df.select("thread_id").first()[0]
).select(
    "post_id", "thread_id", "post_depth",
    "thread_post_count", "instructor_reply_rate"
).show(10, truncate=False)

## Part E — Assemble Final Silver DataFrame + Add Silver Metadata

Add Silver-layer metadata columns:
- _silver_load_ts: when this Silver table was last written
- _schema_version: version tag for schema evolution tracking

Then write to silver.discussion_posts_parsed using OVERWRITE mode.

Why OVERWRITE for Silver?
- This table is fully recomputed from Bronze on every run
- post_depth logic requires seeing all parent posts (can't be incremental easily)
- OVERWRITE guarantees consistency — no duplicate rows from re-runs

In [0]:
# ============================================================
# CELL 14 — Assemble final Silver DataFrame
#           Select and order all output columns cleanly
#           Add Silver metadata columns
# ============================================================

discussion_silver_df = (
    parsed_with_thread_metrics_df
    .select(
        # Core identity columns
        "post_id",
        "thread_id",
        "course_id",
        "author_student_id",
        "parent_post_id",
        # Temporal
        "post_ts",
        # Content metrics
        "content_length_chars",
        "has_attachment",
        "like_count",
        "reply_count",
        "is_instructor_post",
        # Computed Silver columns
        "post_depth",
        "thread_post_count",
        "instructor_reply_rate",
        # Lineage from Bronze
        "_source_file",
        "bronze_load_ts",
    )
    # Add Silver metadata
    .withColumn("_silver_load_ts",  F.current_timestamp())
    .withColumn("_schema_version",  F.lit(SCHEMA_VERSION))
)

print("── Final Silver DataFrame ────────────────────────")
print(f"   Row count : {discussion_silver_df.count():,}")
print(f"   Columns   : {len(discussion_silver_df.columns)}")
print()
discussion_silver_df.printSchema()

In [0]:
# ============================================================
# CELL 15 — Write to silver.discussion_posts_parsed
#           Mode: OVERWRITE — fully recomputed on every run
#           overwriteSchema=true allows schema evolution
# ============================================================

(
    discussion_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DISCUSSION_SILVER_TABLE)
)

final_count = spark.table(DISCUSSION_SILVER_TABLE).count()
print(f"✅ Written to: {DISCUSSION_SILVER_TABLE}")
print(f"   Rows in table: {final_count:,}")

**Expected output:**
```
✅ Written to: mini_project_grp6.silver.discussion_posts_parsed
   Rows in table: 2,000
```

In [0]:
# ============================================================
# CELL 16 — Verify silver.discussion_posts_parsed
# ============================================================

silver_df = spark.table(DISCUSSION_SILVER_TABLE)

print("── discussion_posts_parsed ───────────────────────")
print(f"Total rows   : {silver_df.count():,}")
print(f"Columns      : {len(silver_df.columns)}")
print()

print("── post_depth distribution ───────────────────────")
silver_df.groupBy("post_depth").count().orderBy("post_depth").show()

print("── thread_post_count sample ──────────────────────")
silver_df.select(
    "thread_id", "thread_post_count", "instructor_reply_rate"
).distinct().orderBy(F.desc("thread_post_count")).show(5, truncate=False)

print("── instructor vs student posts ───────────────────")
silver_df.groupBy("is_instructor_post").count().show()

print("── Silver metadata columns ───────────────────────")
silver_df.select(
    "post_id", "_silver_load_ts", "bronze_load_ts", "_schema_version"
).show(3, truncate=50)

## Part F — Data Quality Checks (Silver)

Silver DQ is stricter than Bronze:
1. post_depth must only contain values 0, 1, or 2
2. thread_post_count must be >= 1 for every row
3. instructor_reply_rate must be between 0.0 and 1.0
4. course_id and author_student_id must not be NULL

All failures written to audit.dq_audit with severity tags.

In [0]:
# ============================================================
# CELL 18 — DQ Check 1: post_depth must be 0, 1, or 2
# Any NULL or out-of-range value indicates a parsing error
# ============================================================

silver_df = spark.table(DISCUSSION_SILVER_TABLE)

invalid_depth = (
    silver_df
    .filter(
        F.col("post_depth").isNull() |
        (~F.col("post_depth").isin(0, 1, 2))
    )
    .withColumn("dq_check_name", F.lit("invalid_post_depth"))
    .withColumn("dq_table",      F.lit(DISCUSSION_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("HIGH"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("post_depth out of range [0,1,2]:"),
        F.col("post_depth").cast("string"),
        F.lit("post_id:"), F.col("post_id")
    ))
)

invalid_depth_count = invalid_depth.count()
print("── DQ Check 1: post_depth values ─────────────────")
print(f"   Invalid post_depth rows: {invalid_depth_count}")

if invalid_depth_count > 0:
    (
        invalid_depth
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "post_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_depth_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    invalid_depth.select("post_id", "post_depth", "dq_message").show(5, truncate=False)
else:
    print("   ✅ PASSED — all post_depth values are 0, 1, or 2")

In [0]:
# ============================================================
# CELL 19 — DQ Check 2a: thread_post_count must be >= 1
#           DQ Check 2b: instructor_reply_rate must be in [0.0, 1.0]
# ============================================================

# ── 2a: thread_post_count ──────────────────────────────────
invalid_thread_count = (
    silver_df
    .filter(
        F.col("thread_post_count").isNull() |
        (F.col("thread_post_count") < 1)
    )
    .withColumn("dq_check_name", F.lit("invalid_thread_post_count"))
    .withColumn("dq_table",      F.lit(DISCUSSION_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("thread_post_count < 1 for thread_id:"),
        F.col("thread_id")
    ))
)

tc_count = invalid_thread_count.count()
print("── DQ Check 2a: thread_post_count ────────────────")
if tc_count > 0:
    (
        invalid_thread_count
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "post_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {tc_count} rows flagged → written to {DQ_AUDIT_TABLE}")
else:
    print(f"   ✅ PASSED — all thread_post_count >= 1")

# ── 2b: instructor_reply_rate ──────────────────────────────
invalid_reply_rate = (
    silver_df
    .filter(
        F.col("instructor_reply_rate").isNull() |
        (F.col("instructor_reply_rate") < 0.0) |
        (F.col("instructor_reply_rate") > 1.0)
    )
    .withColumn("dq_check_name", F.lit("invalid_instructor_reply_rate"))
    .withColumn("dq_table",      F.lit(DISCUSSION_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("instructor_reply_rate out of [0,1]:"),
        F.col("instructor_reply_rate").cast("string"),
        F.lit("thread_id:"), F.col("thread_id")
    ))
)

irr_count = invalid_reply_rate.count()
print()
print("── DQ Check 2b: instructor_reply_rate ────────────")
if irr_count > 0:
    (
        invalid_reply_rate
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "post_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {irr_count} rows flagged → written to {DQ_AUDIT_TABLE}")
else:
    print(f"   ✅ PASSED — all instructor_reply_rate values in [0.0, 1.0]")

In [0]:
%sql
-- ============================================================
-- CELL 21 — SQL verification of silver.discussion_posts_parsed
-- ============================================================

-- Row counts and key metrics summary
SELECT
    COUNT(*)                                      AS total_posts,
    COUNT(DISTINCT thread_id)                     AS unique_threads,
    COUNT(DISTINCT course_id)                     AS unique_courses,
    COUNT(DISTINCT author_student_id)             AS unique_students,
    SUM(CASE WHEN post_depth = 0 THEN 1 ELSE 0 END) AS top_level_posts,
    SUM(CASE WHEN post_depth = 1 THEN 1 ELSE 0 END) AS depth_1_replies,
    SUM(CASE WHEN post_depth = 2 THEN 1 ELSE 0 END) AS depth_2_replies,
    SUM(CASE WHEN is_instructor_post = true THEN 1 ELSE 0 END) AS instructor_posts,
    ROUND(AVG(thread_post_count), 2)              AS avg_posts_per_thread,
    ROUND(AVG(instructor_reply_rate), 4)          AS avg_instructor_reply_rate
FROM mini_project_grp6.silver.discussion_posts_parsed;


**Expected output:**
```
+------------+--------------+---------------+-----------------+-----------------+----------------+----------------+-----------------+--------------------+--------------------------+
|total_posts |unique_threads|unique_courses |unique_students  |top_level_posts  |depth_1_replies |depth_2_replies |instructor_posts |avg_posts_per_thread|avg_instructor_reply_rate |
+------------+--------------+---------------+-----------------+-----------------+----------------+----------------+-----------------+--------------------+--------------------------+
|2,000       |200           |~220           |~580             |~600             |~900            |~500            |~250             |~10.0               |~0.12                     |
+------------+--------------+---------------+-----------------+-----------------+----------------+----------------+-----------------+--------------------+--------------------------+
```

In [0]:
# ============================================================
# CELL 22 — Task 2.1 completion summary
# ============================================================

final_count = spark.table(DISCUSSION_SILVER_TABLE).count()

print("═" * 58)
print("  TASK 2.1 COMPLETE — Silver Discussion Posts Parsed")
print("═" * 58)
print()
print(f"  ✅ {DISCUSSION_SILVER_TABLE}")
print(f"      Rows     : {final_count:,}")
print(f"      Source   : {DISCUSSION_BRONZE_TABLE}")
print(f"      Mode     : OVERWRITE (fully recomputed)")
print()
print("  Computed columns added in Silver:")
print("      post_depth           — 0 (top), 1 (reply), 2 (reply-to-reply)")
print("      thread_post_count    — total posts per thread (Window)")
print("      instructor_reply_rate — instructor fraction per thread (Window)")
print()
print("  DQ checks run:")
print("      Cell 18 — post_depth must be 0, 1, or 2")
print("      Cell 19 — thread_post_count >= 1")
print("      Cell 19 — instructor_reply_rate in [0.0, 1.0]")
print("      Cell 20 — NULL course_id / author_student_id")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 2.2 → 05_silver_learning_sessions")
print("         LAG window + 30-min gap detection + session_id")
print("═" * 58)